# 灵巧手的步态策略训练

借鉴 `Sim-to-Real Learning of All Common Bipedal Gaits via Periodic Reward Composition` 的核心思想，用**周期性奖励组合**而不是参考轨迹，去训练能够在手内重定向任务中形成稳定 finger gait、并且能够在多种 finger gait 之间切换的单一策略。

## 1. 这篇 paper 最值得迁移的是什么

这篇 paper 的关键不是“学会双足走路”，而是提出了一种**描述周期行为的奖励设计语言**：

1. 不直接给参考轨迹，而是给**周期相位上应该出现/不应该出现的物理现象**。
2. 对双足来说，这个物理现象非常简单：
   - 摆动期：不应该有脚底力，可以有脚速度。
   - 支撑期：不应该有脚速度，可以有脚底力。
3. 用相位指示函数 $I(\phi)$ 在周期内打开/关闭这些奖励项。
4. 用带不确定性的相位边界（文中用 Von Mises 分布）把 phase boundary 软化，避免硬切换导致训练不稳定。
5. 用少量可解释参数去描述 gait：
   - 相位长度比 $r$
   - 左右脚相位偏移 $\theta$
6. 把这些 gait 参数同时喂给 policy，于是 policy 学到的不只是一个 gait，而是一个**gait-conditioned controller**。
7. 当多 gait 混合训练时，仅靠基础周期奖励不够，还要加**transition penalties** 来排除不希望的混合行为。

我对这篇 paper 的一句话理解：

> 它把“步态”从轨迹模仿问题，变成了“在什么时候让哪些物理量出现/消失”的问题。

这点对手指步态尤其有价值，因为手指 gait 本来就很难写成唯一的参考轨迹，但很好写成“接触/卸载/重抓取”的周期规律。

## 2. 对手指步态的直接类比

双足 locomotion 和手内 finger gait 的映射关系可以写成：

- foot stance $\leftrightarrow$ 某根手指处于 **support/load-bearing contact**
- foot swing $\leftrightarrow$ 某根手指处于 **release/reposition/regrasp**
- forward velocity command $\leftrightarrow$ 期望的 **物体重定向角速度/连续旋转指令**
- pelvis stability $\leftrightarrow$ **物体稳定性**（不掉落、不大幅漂移、不异常翻滚）
- gait family $\leftrightarrow$ **contact handoff pattern**（哪根手指什么时候支撑、什么时候换位）

如果把 hand gaiting 看成“在保持物体稳定的同时，周期性地重新分配接触点和支撑载荷”，那么它和 biped gait 的结构非常接近：

- locomotion 中交替的是两只脚的支撑责任
- in-hand gaiting 中交替的是多根手指的支撑责任

因此，一个自然的 hypothesis 是：

> 对连续重定向任务，finger gait 可以被描述为每根手指在一个周期内交替进入 support phase 和 reposition phase；策略只需被告知当前 phase 和 gait 参数，就能学到不同 contact handoff 模式。

## 3. 为什么不能生搬硬套，而要做一点改造

这里有一个重要差别：

- 对脚来说，“离地”几乎天然意味着“正在移动去下一落脚点”。
- 对手指来说，“失去接触”并**不一定**意味着“正在朝更好的接触点移动”。它也可能只是悬空、抖动、或者让别的手指代偿。

所以对 finger gaiting 来说，只用二相位 `support/reposition` 可能不够，至少要考虑下面两种设计：

### 方案 A：最小可行二相位

- support phase：鼓励接触、承载、低滑移、低相对运动
- reposition phase：鼓励卸载，允许移动，但不强迫具体路径

优点：
- 最接近论文原始思路
- 最容易先做出结果

缺点：
- finger 可能学会“消极悬空”而不是“主动重抓取”

### 方案 B：更适合手指的三相位/四相位

例如：
- support
- release
- reach
- regrasp

优点：
- 更符合手指换接触点的真实过程
- 更容易把“什么时候应该重新建立接触”写清楚

缺点：
- 奖励设计和 phase command 都会更复杂

我的判断是：

> 第一版应该先做二相位，把周期 reward composition 跑通；如果出现“悬空不重抓”“所有手指一起滑”“靠 palm 作弊”等退化策略，再升级到三相位或加 transition penalties。

## 4. 适合 AnyMani 的形式化定义

设有 $N_f$ 根参与 gait 的手指，主时钟相位为 $\phi \in [0,1)$。对每根手指 $i$，定义：

- support phase 系数 $C_i^{\text{sup}}(\phi)$
- reposition phase 系数 $C_i^{\text{rep}}(\phi)$
- 相位偏移 $\theta_i$
- support ratio $r_i$

与论文一致，相位边界不做硬开关，而是用软边界：

$$
C_i^{\text{sup}}(\phi), C_i^{\text{rep}}(\phi) \in [0,1]
$$

并满足它们在大多数时刻近似互补。这样 reward 在 phase transition 附近更平滑。

### 4.1 每根手指的基础测量量

对第 $i$ 根手指，可以定义：

- $f_i(s)$：指尖接触合力大小
- $b_i(s)$：指尖是否接触的二值量
- $v_i(s)$：指尖 twist 或速度大小
- $\ell_i(s)$：该指尖承担的支撑/载荷贡献
- $\sigma_i(s)$：接触滑移量或切向相对速度

其中当前 AnyMani 里已经比较容易拿到的是：

- `fingertip_contact_data(..., output_type="force")` 对应连续接触力
- `fingertip_contact_data(..., output_type="binary")` 对应二值接触状态
- `body_twists` 对应手指末端的 twist
- `bad_palm_contact` / `good_fingertip_contact` / `load_distribution_reward` 可作为现成的接触 shaping
- `track_rotation_velocity_alignment` 可作为连续旋转任务的执行奖励

### 4.2 一个可落地的 reward 草案

令：

- $r_{\text{task}}(s)$：物体沿命令轴旋转的执行奖励
- $q_{\text{fall}}(s)$：掉落/偏离惩罚
- $q_{\text{smooth}}(s)$：动作平滑惩罚
- $q_{\text{energy}}(s)$：能量/力矩惩罚
- $q_{\text{palm}}(s)$：非期望 palm 或非指尖接触惩罚

则总 reward 可以写成：

$$
R(s,\phi)= w_{\text{task}} r_{\text{task}}(s)
- w_{\text{fall}} q_{\text{fall}}(s)
- w_{\text{smooth}} q_{\text{smooth}}(s)
- w_{\text{energy}} q_{\text{energy}}(s)
- w_{\text{palm}} q_{\text{palm}}(s)
+ \sum_{i=1}^{N_f} R_i^{\text{gait}}(s,\phi)
$$

其中每根手指的 gait reward 可以先写成：

$$
R_i^{\text{gait}}(s,\phi)=
C_i^{\text{sup}}(\phi)
\Big(
+w_f \hat f_i
- w_v \|v_i\|
- w_{\text{slip}} \sigma_i
\Big)
+
C_i^{\text{rep}}(\phi)
\Big(
-w_f \hat f_i
- w_{\text{stall}} e^{-\|v_i\|}
\Big)
$$

这里：

- support phase 中：希望该手指有接触/有载荷，但不要乱动、不要滑
- reposition phase 中：希望该手指先卸载，且不要停滞不动

如果第一版不想加 $\sigma_i$，可以先省略滑移项，只保留：

$$
R_i^{\text{gait}}(s,\phi)=
C_i^{\text{sup}}(\phi)(+w_f \hat f_i - w_v\|v_i\|)
+
C_i^{\text{rep}}(\phi)(-w_f \hat f_i)
$$

这是最接近 paper 原始“支撑期保力、摆动期去力”的写法。

### 4.3 对 hand 来说必须额外加的约束

和双足不同，finger gait 很容易出现下面这些退化行为：

1. 所有手指同时松开或同时滑动
2. 一根手指在它不该接触的时候仍然粘着物体
3. 用 palm 或近端 link 偷偷承载
4. 策略干脆学成静态抓取，根本没有真正的 gait

因此我认为至少还需要加三类附加项：

#### a. 最小支撑数约束

$$
q_{\min\_support}(s)=\big[N_{\min}-\sum_i b_i(s)\big]_+^2
$$

用于保证任意时刻仍有足够多的支撑手指。

#### b. 非计划接触惩罚

如果某根手指当前处于 reposition window，却长期维持高接触，那就说明 schedule 没有真正生效。

$$
q_{\text{unscheduled-contact}}(s,\phi)=
\sum_i C_i^{\text{rep}}(\phi) \cdot b_i(s)
$$

#### c. handoff / regrasp 奖励

当某根手指在预定窗口重新建立接触时，应该给额外奖励，否则它可能只学会“松开”而不会“回来”。

一个简单版本可以写成：

$$
r_{\text{regrasp}}(s,\phi)=
\sum_i C_i^{\text{touch}}(\phi) \cdot \mathbf{1}[b_i(s)=1]
$$

其中 $C_i^{\text{touch}}$ 可以是 reposition phase 末端的一小段窗口。

这个项我认为对手指非常关键，它相当于 paper 里 multi-gait controller 所依赖的 transition penalties 在 hand setting 下的对应物。

## 5. gait 参数化应该怎么设计

对双足来说，gait 由 phase ratio 和左右脚 offset 决定。对 4 指 LeapHand，可以把 gait 参数推广成：

- 每根手指的 support ratio $r_i$
- 每根手指的相位偏移 $\theta_i$
- 可选的 regrasp window 宽度 $\delta_i$

这样可以表达多种 finger gait：

### 5.1 Traveling-wave gait

例子：thumb $\to$ index $\to$ middle $\to$ ring 依次换位。

特征：
- 任一时刻只有 1 根手指在 reposition
- 其余手指持续 support
- 最适合作为第一版稳定 gait

### 5.2 2-2 交替 gait

例子：两根手指组成一组交替做 contact handoff。

特征：
- 节奏更强
- 对物体稳定性的要求更高
- 更容易出现“同时失稳”

### 5.3 Static grasp / standing 类比

像 paper 里把 standing 当成 gait family 的一个极端情况一样，hand 里也可以把“静态稳定抓持”看成 $r_i \to 1$ 的退化极限。

这样单一策略就有机会从：

- 静态抓持
- 慢速 finger gait
- 高频 finger gait

之间连续切换。

## 6. 对 policy observation / command 的启发

paper 有一个很关键的做法：**reward 的 phase 参数必须进入 observation**，否则环境对策略来说不是 Markov 的。

迁移到 AnyMani 后，我建议 policy 至少额外接收：

- 全局时钟编码：$\sin(2\pi \phi), \cos(2\pi \phi)$
- 每根手指的 phase-offset clock：$\sin(2\pi(\phi+\theta_i)), \cos(2\pi(\phi+\theta_i))$
- gait ratio 向量：$[r_1,\dots,r_{N_f}]$
- 可选的 regrasp window 参数

和当前 AnyMani 已有观测拼起来，student policy 可以是：

- `body_twists`
- `fingertip_contact_binary`
- `so3_command`
- `last_action`
- `phase/gait command`

critic 则继续保留：

- `object_pos`, `object_quat`
- `fingertip_contact_force`
- 其他特权状态

这和 paper 的 spirit 是一致的：

> 策略不知道未来轨迹，但知道当前自己在 gait cycle 的什么位置，以及当前命令的是哪种 gait。

## 7. AnyMani 里哪些现成模块可以直接复用

当前代码里已经有不少现成积木，可以直接承接这个 idea：

### 现有可直接复用

- `body_twists`：可以度量每根手指末端的运动强度
- `fingertip_contact_data(binary/force)`：可以直接做接触存在性和接触力度
- `good_fingertip_contact`：可以鼓励最少支撑接触数
- `bad_palm_contact`：可以避免 palm 作弊
- `load_distribution_reward`：可以做“支撑责任分配”的初步 proxy
- `track_rotation_velocity_alignment`：很适合 continuous rotation / redirection
- `fall_penalty`：保证物体别掉
- `pose_diff_penalty` / `se3_kinetic_energy` / `se3_action_smooth`：可直接沿用作 regularization

### 大概率需要新增的 reward / observation

1. **per-finger scheduled contact reward**
   - 不只是“总共有多少个 contact”，而是“该接触的那根手指有没有在该时刻接触”
2. **per-finger unload reward**
   - 在 reposition window 明确鼓励某根手指卸载
3. **minimum-support-count penalty**
   - 保证换位过程中抓持不崩
4. **regrasp event bonus**
   - 明确奖励在预定窗口重新建立 contact
5. **relative slip / tangential velocity**
   - 如果只看接触有无，不足以区分“稳定支撑”与“带着滑移的假支撑”

我觉得第 1、3、4 项是最先该补的；第 5 项是后续把 gait 做“像样”的关键项。

## 8. 训练路线建议

### Stage 1: 单 gait，先证明周期 reward 对手指有效

任务设定建议：

- 物体：round / capsule / 轴对称物体
- 指令：连续绕固定轴旋转，而不是 one-shot orientation matching
- gait：固定一个 traveling-wave schedule
- goal：看策略是否出现稳定的 sequential contact handoff

这里最重要的不是最终 rotation rate，而是：

- 是否出现清晰的 phase-locked finger contact pattern
- 是否能在不掉物体的前提下完成持续重定向

### Stage 2: gait-conditioned 单策略

开始随机化：

- $r_i$
- $\theta_i$
- rotation speed command

把 gait 参数作为 observation 注入，让策略学习一族 gait，而不是一个 gait。

### Stage 3: 多 gait + transition penalties

这一步要重点防止：

- 所有手指滑成一团
- 只学会 static grasp
- 学会用 palm 顶住物体假装完成任务

这里需要 hand 版的 transition penalties，例如：

- hop symmetry 对应的 **opposition geometry / contact span constraint**
- standing penalty 对应的 **static grasp regularizer**
- scheduled regrasp bonus

### Stage 4: sim-to-real / object 泛化

和 paper 一样，真正落地必须做 domain randomization：

- 物体质量、COM、摩擦
- 接触阈值噪声
- actuator gain / damping
- 观测噪声
- 手指 link 接触摩擦

但我认为在 hand 任务里，sim-to-real 还有一个额外难点：

> 接触事件本身比双足“是否着地”更加敏感，tactile threshold 和摩擦模型误差会更直接地破坏 gait timing。

所以 phase boundary 的软化在这里可能比 locomotion 更重要。

## 9. 我认为最值得先验证的 MVP

如果现在就开始做，我会选这个最小实验：

### 任务

- 使用 `Se3Tactile` 路线
- 连续绕单一轴做 rolling / continuous rotation
- 只在圆柱或近轴对称物体上做

### 动作与观测

- 动作：沿用现有 se(3) fingertip action
- 观测：`body_twists + tactile binary + so3 command + phase clocks + gait params`

### 奖励

- 主任务：`track_rotation_velocity_alignment`
- 稳定项：`fall_penalty + bad_palm_contact + action smooth + kinetic energy`
- gait 项：
  - scheduled support finger 应该接触/承载
  - scheduled reposition finger 应该卸载
  - minimum support count
  - regrasp bonus

### 成功指标

除了 task return 之外，我会额外记录：

- 每根手指 contact 的 phase histogram
- schedule agreement：某手指在自己 support window 内接触的比例
- unscheduled contact ratio
- 重抓取成功率
- rotation rate / drop rate
- 不同 gait command 下的切换成功率

如果这一步能做出来，我们就能比较有说服力地说：

> finger gait 不是训练中偶然出现的接触模式，而是可以被显式参数化、条件化和调用的 manipulation primitive。

## 10. 这个 idea 最核心的科研价值

我觉得它的价值不只是“让手指换着抓”，而是提供一个新的研究问题表述：

> 把 dexterous in-hand manipulation 中的 contact rearrangement，提升为一种可编程的 gait prior。

这可能带来三层贡献：

1. **方法层**：把 periodic reward composition 从 legged locomotion 推广到 dexterous contact scheduling。
2. **策略层**：学到一个 gait-conditioned manipulation policy，而不是一个单一抓法。
3. **科学问题层**：回答“手内操作中的 finger gait 是否也存在类似 locomotion gait family 的低维结构”。

## 11. 我当前的判断

我目前比较认可的路线是：

- 不要一开始追求“所有对象、所有 gait、全部泛化”
- 先把 **continuous rotation + traveling-wave finger gait** 做成
- 先验证“周期相位奖励真的能让 contact handoff 变得可控”
- 再去谈 multi-gait 和 sim-to-real

原因很简单：

> 这个 idea 的第一性问题不是“最后能转多快”，而是“phase-conditioned reward 能否在手内操作里产生稳定、可解释、可调参的接触时序结构”。

只要这个问题回答清楚，后面很多扩展都顺了。

## 12. 下一轮我们最该讨论的 3 个问题

1. 第一版到底用 **二相位** 还是直接上 **三相位/四相位**？
2. 第一版的主任务应该用 **fixed-goal reorientation** 还是 **continuous rotation**？
3. 我们把“support”定义成 **接触存在**、**接触力超过阈值**，还是 **低滑移承载接触**？

我现在倾向于：

- 二相位起步
- continuous rotation 作为主任务
- support 先用“接触力超过阈值”定义，后续再加入 slip 约束

因为这样最接近 paper 的原始精神，也最容易做出第一版验证。

## 13. 关于创新性的再判断：不要把 hand 硬类比成 bipeds

用户反馈里提到一个非常关键的问题：

> 手内旋转里可稳定涌现的 finger gait 可能并不多，很多时候只是一个近乎固定的循环顺序，例如 `thumb -> ring -> middle -> index` 或其逆序；如果是这样，那么把问题包装成“像双足一样存在 walking/running/hopping/skipping 那样丰富的 gait family”可能并不成立。

我认同这个担心，而且我认为这里恰恰应该**主动收缩 claim**，否则论文定位会虚。

### 13.1 这件事为什么不一定是坏事

双足 locomotion 的 gait family 丰富，是因为：

- 身体形态高度对称
- 任务目标比较统一（产生身体推进）
- 支撑切换的自由度相对有限但又足够形成多种节律

而 finger gaiting 不一样：

- hand 的形态天然不对称，thumb 的角色特殊
- object geometry、重力方向、摩擦条件对稳定 contact order 的约束很强
- 任务不是“推进身体”，而是“在不掉物体的情况下持续重分配接触并产生期望物体转动”

所以一个完全可能、而且我觉得更真实的结论是：

> 手内 finger gait 的稳定策略空间不是“若干离散 gait 家族”，而是一个**很低维、强约束**的 contact scheduling manifold；其中循环顺序可能几乎固定，真正能变化的是 duty factor、节奏、是否跳过某根手指、以及何时触发 handoff。

如果实验支持这一点，它并不是负结果，反而可能是更有洞察力的结论。

### 13.2 因此创新点不该放在哪里

我认为**不要**把创新点放在：

- “提出了手指版 walking/running/hopping/skipping taxonomy”
- “发现了很多种像双足一样的通用手指步态”

这两个说法都风险很大，也不一定真实。

### 13.3 更合理的创新点应该放在哪里

更稳、更有科研味道的定位是：

1. **从‘学会一个 emergent finger gait’转向‘显式参数化和调制 contact schedule’**
   - 不是说 hand 一定有很多离散 gait
   - 而是说我们可以把接触切换的时序结构变成一个可控变量

2. **把 periodic reward composition 从 locomotion 迁移到 dexterous contact scheduling**
   - 贡献点在 reward language 和 control prior
   - 不是在 gait taxonomy 的数量

3. **证明 finger gaiting 的低维结构可以被 policy conditioning 捕获**
   - 即使只有一个主循环顺序，也可能存在：
     - 慢速高 duty factor 版本
     - 快速低 duty factor 版本
     - 跳过某根手指的稀疏版本
     - 顺/逆时针两种方向版本

4. **把‘phase management’问题学习化**
   - 经典 dexterous manipulation 文献里本来就有 event-driven / phase transition 的中层控制问题
   - 我们可以把它做成基于 RL 的、可微调的、可条件化的 phase prior

5. **如果最后实验发现 order 基本唯一，这本身也可以成为结果**
   - 说明在给定 hand morphology 和 object family 下，稳定 finger gait 不是一个丰富离散集，而是一个窄的可行域
   - 这时论文贡献可以是“发现并利用这个可行域”，而不是“枚举很多 gait”

### 13.4 已有相关工作的提醒

外部文献里已经有非常接近的问题设定：

- `On the Feasibility of Learning Finger-gaiting In-hand Manipulation with Intrinsic Sensing`（ICRA 2022）
  - 他们已经做了：
    - 用 RL 学 finger-gaiting
    - 只用 intrinsic sensing（proprio + tactile）
    - 连续绕轴重定向
    - 重点解决 exploration / initial state distribution

这意味着：

> 如果我们只是做“用 RL + tactile/proprio 学会 finger gaiting”，创新性明显不够。

所以我们必须把问题往前推进一步。

### 13.5 我认为可成立的论文定位

如果继续做，我更建议把标题级别的问题改成下面这种：

- **Phase-Conditioned Contact Scheduling for In-Hand Finger Gaiting**
- **Periodic Reward Composition for Dexterous Contact Handoffs**
- **Learning Low-Dimensional Finger-Gait Manifolds for In-Hand Reorientation**

注意这几种说法都没有承诺“存在很多像双足那样的 gait family”。

它们真正强调的是：

- contact handoff 是可编程的
- schedule 是可调的
- 结构是低维的
- RL 能学到这个结构

### 13.6 因此实验问题也要改

与其问：

- “能不能学出 walking/running/hopping/skipping 式的多个 hand gait？”

不如问：

- “稳定 finger gait 的可行时序空间有多大？”
- “phase conditioning 是否能把 emergent contact sequence 变成可控 schedule？”
- “policy 是否能在保持同一循环顺序的情况下，连续调节 duty factor / cadence / skipped finger pattern？”
- “clock prior 和 event-driven phase transition 哪个更适合 hand？”

### 13.7 一个更稳的主张

我目前认为最稳的 scientific claim 是：

> 对 hand 来说，重要的也许不是存在多少个离散 gait，而是存在一个可学习、可条件化、可解释的 contact handoff manifold；周期性奖励组合提供了一种把这个 manifold 显式注入策略学习的方法。

这个 claim 比“手也有很多 common gaits”要稳得多，也更符合你现在的直觉和经验观察。

### 13.8 这会如何改变我们下一步实现

如果接受上面的定位，那么第一版实现就不用追求“多 gait catalogue”，而应该追求：

1. 学出一个稳定的主循环顺序
2. 让这个顺序的 duty factor / phase width / cadence 可调
3. 验证这种 schedule conditioning 比无结构 RL 更可控、更稳、更高效
4. 研究 object family / gravity direction / rotation axis 改变时，可行时序空间怎么变

这比硬凑多个 gait 名称更像一篇扎实的 manipulation paper。

# 我的想法
## 创新性
根据以前训练结果，和自己用手内转动物体的表现，我发现好像能涌现的步态并不多，基本是thumb -> ring -> middle -> index这种顺序（或反向）的循环，双足机器人的步态似乎通过 $ r $ 和 $\theta$ 能涌现出running，walking，hopping，skipping等步态，但我实在想像不出灵巧手的步态有哪些。以手内旋转为例，似乎只有thumb -> ring -> middle -> index这种的循环，或者再不济少用一颗手指，但顺序还是这样。因此反而没有running，walking，hopping，skipping等步态，因为手内操作的步态是在训练任务中自动涌现的。我担心咱们的创新性不足